# Experiment B — Reconstructed Acceleration Through the Frozen Experiment A CNN

**Question:** how much of the representation learned from raw acceleration survives when the same frozen model is fed event-derived reconstructed acceleration?

Strict protocol:

1. Load the authoritative **Experiment A checkpoint**.
2. Reuse its exact **model weights, user split, `class_to_idx`, and raw-train normalization**.
3. Do **not** train or fine-tune the CNN on reconstruction.
4. Do **not** refit normalization on reconstruction.
5. Keep downstream references raw: raw train gallery, raw validation for kNN K selection, raw train prototypes, and raw train/validation for the linear probe.
6. Evaluate both matched held-out views with the same frozen feature extractor:
   - raw test → baseline/reference consistency,
   - reconstruction test → Experiment B.
7. Run paired raw/reconstruction embedding and prediction preservation diagnostics.
8. Use the same selected dataset roots as the A checkpoint; a root/action/sample mismatch must be resolved by regenerating A.

This notebook is intentionally thin. The executable workflow lives in `scripts/run_experiment_b.py`; reusable implementation lives in `snn/accel_reconstruction_eval/`.


In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
from pathlib import Path
import sys

import pandas as pd
import torch


def find_repository_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "snn").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate repository root containing 'snn' and 'scripts'."
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd())
SCRIPTS_DIR = REPOSITORY_ROOT / "scripts"

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from snn.accel_reconstruction_eval import experiment_b_config
from run_experiment_b import run_experiment_b

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

print("Repository root:", REPOSITORY_ROOT)
print("CUDA available:", torch.cuda.is_available())


## Paths and runtime controls

`BASELINE_A_CHECKPOINT` is authoritative. Experiment B never updates it and never writes a replacement model checkpoint.

Regenerate Experiment A before running B/C/D whenever A's exclusions change. B has no local cohort or explicit-split override.


In [ ]:
# Select the same one or two roots used to produce BASELINE_A_CHECKPOINT.
# Roots remain separate on disk and are combined only in the shared loader.
# The runner owns model/training logic; this is the only probe switch needed here.
PROBE_VARIANT = "cnn_l"  # one of: cnn_s, cnn_m, cnn_l
DATASET_ROOTS = [
    Path("outputs/action0_rectified/low-pass/aligned-board-events"),
    Path("outputs/action1_rectified/low-pass/aligned-board-events"),
]
BASELINE_A_CHECKPOINT = Path(
    "notebooks/artifacts/acceleration_cnn_representation/"
    f"{PROBE_VARIANT}/best_acceleration_cnn.pt"
)
OUTPUT_DIR = Path(
    "notebooks/artifacts/experiment_B_reconstruction_frozen_cnn"
)
if not BASELINE_A_CHECKPOINT.is_file() and PROBE_VARIANT == "cnn_l":
    BASELINE_A_CHECKPOINT = Path(
        "notebooks/artifacts/acceleration_cnn_representation/best_acceleration_cnn.pt"
    )

RANDOM_SEED = 12345
BATCH_SIZE = 128
NUM_WORKERS = 0
USE_GPU = True


## Build the Experiment B configuration

The domain contract is fixed:

- train/reference = raw,
- validation/reference = raw,
- test/query = reconstruction,
- normalization = loaded from the Experiment A checkpoint,
- CNN training = disabled.


In [ ]:
config = experiment_b_config(
    baseline_checkpoint=BASELINE_A_CHECKPOINT,
    output_dir=OUTPUT_DIR,
    random_seed=RANDOM_SEED,
    probe_variant=PROBE_VARIANT,
)
config = replace(
    config,
    use_gpu=USE_GPU,
    loader=replace(
        config.loader,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    ),
    evaluation=replace(
        config.evaluation,
        random_seed=RANDOM_SEED,
    ),
)
config.validate()
config.to_dict()


## Run Experiment B

This call performs:

`load A checkpoint → validate reconstruction packages → recover A split/class mapping → load A normalization → strict frozen restore → raw/reconstruction embedding extraction → shared evaluation → paired preservation → save standardized artifacts`


In [ ]:
run = run_experiment_b(
    root=DATASET_ROOTS,
    repository_root=REPOSITORY_ROOT,
    output_dir=OUTPUT_DIR,
    baseline_checkpoint=BASELINE_A_CHECKPOINT,
    config=config,
    probe_variant=PROBE_VARIANT,
)


## Inherited cohort provenance

Experiment B inherits this cohort from the authoritative A checkpoint; no user-cohort controls are defined locally.


In [ ]:
provenance = json.loads(
    (run.output_dir / "provenance.json").read_text(encoding="utf-8")
)
display(
    pd.Series(
        {
            "cohort_source": provenance["cohort_source"],
            "excluded_users": provenance["excluded_users"],
            "eligible_users": provenance["eligible_users"],
            "train_users": provenance["train_users"],
            "val_users": provenance["val_users"],
            "test_users": provenance["test_users"],
            "class_to_idx": provenance["class_to_idx"],
        },
        name="Inherited cohort",
    )
)


## Experiment B primary result

The root `summary.csv` corresponds to **reconstruction test through the frozen A model**.


In [ ]:
display(run.summary.T.rename(columns={0: "Experiment B"}))


## Raw-test reference

This is the same frozen checkpoint and normalization evaluated on raw held-out test inputs. It should remain consistent with the standardized Experiment A result, up to deterministic numerical tolerance.


In [ ]:
display(run.raw_reference_summary.T.rename(columns={0: "A raw reference"}))


## Raw vs reconstruction comparison


In [ ]:
display(run.domain_comparison)


## Frozen-CNN classification across domains


In [ ]:
display(run.classification_splits)


## Split contract


In [ ]:
display(run.split_summary)
display(run.label_split_counts)


## Checkpoint normalization

These values are loaded from Experiment A; they are not re-estimated from reconstruction.


In [ ]:
pd.DataFrame(
    {
        "channel": ["x", "y", "z"],
        "mean": run.normalization.mean,
        "std": run.normalization.std,
    }
).assign(
    fitted_on=run.normalization.fitted_on,
    valid_time_points=run.normalization.valid_time_points,
)


## Paired raw/reconstruction preservation


In [ ]:
display(run.paired_summary)
display(run.paired_transitions)


## Baseline checkpoint identity


In [ ]:
pd.DataFrame(
    {
        "field": ["path", "sha256"],
        "value": [
            str(run.baseline_checkpoint_path),
            run.baseline_checkpoint_sha256,
        ],
    }
)


## Saved artifacts


In [ ]:
artifact_table = pd.DataFrame(
    [
        {"artifact": name, "path": str(path)}
        for name, path in sorted(run.artifact_paths.items())
    ]
)
display(artifact_table)


# Visualization

The cells below are presentation-only. They consume the metrics and embeddings already generated by `run_experiment_b()`.

In [ ]:
from snn.accel_reconstruction_eval.visualization import visualize_experiment_b

figures = visualize_experiment_b(run.output_dir)

## Interpretation

Use Experiment B together with A and C:

- **A high, B low, C ≈ A:** reconstructed acceleration still contains recoverable task information, but the raw-trained representation suffers domain shift.
- **A high, B low, C low:** reconstruction likely removes or strongly distorts task-relevant information.
- High paired embedding cosine similarity and prediction agreement support representation preservation; low values localize the raw→reconstruction change.
- Do not interpret a B accuracy drop alone as proof of information destruction, because B intentionally forbids adaptation to the reconstruction domain.
